# STEP11. MRL + DMD 눈 데이터셋 통합

## 분석 질문

- MRL(대량·IR·균형)과 DMD(소량·RGB·불균형)를 어떤 표본 비율과 어떤 분할로 합치면, 합본이 MRL 단독보다 Closed 탐지에 유리한 학습 데이터가 되는가.
- 관련 가설: H1 — 두 데이터셋을 클래스 균형을 유지한 채 합치면 DMD hold-out subject 에서의 Closed-Recall 이 MRL 단독보다 높다. (검정은 STEP12)

## 입력

| 구분 | 경로 |
|---|---|
| MRL subject 분할 manifest | `config.OUTPUTS_DIR / "mrl_split" / "mrl_manifest_subject.csv"` |
| MRL 눈 이미지 | `build_dmd_eye_dataset.data_subdir("MRL Eye") / "data"` |
| DMD 프레임 GT (STEP10) | `config.OUTPUTS_DIR / "dmd_gt"` |
| DMD mosaic 영상 | `build_dmd_eye_dataset.dmd_dir()` |
| YuNet 얼굴 검출 모델 | `config.YUNET_MODEL` (미병합 시 `model/detectors/` fallback) |

## 출력

| 파일 | 위치 |
|---|---|
| DMD 눈 crop png | `config.OUTPUTS_DIR / "dmd_eye" / <split> / <label>` |
| `dmd_eye_manifest.csv` | `config.OUTPUTS_DIR / "dmd_eye"` |
| `eye_manifest.csv` (통합) | `config.OUTPUTS_DIR / "eye_dataset"` |
| `class_balance.csv` | `config.OUTPUTS_DIR / "eye_dataset"` |
| `leakage_report.csv` | `config.OUTPUTS_DIR / "eye_dataset"` |

## 전체 수행 흐름

**PART A — 진단: 그냥 합치면 무엇이 깨지는가**

1. 설정·경로
2. MRL·DMD 클래스 구성 정량화
3. 단순 합본의 클래스 비율 계산

**PART B — DMD 눈 crop 생성**

4. subject 분할과 클래스별 stride 확정
5. 예상 표본 수 확인 (영상 디코딩 전)
6. 눈 crop 생성

**PART C — 통합 manifest**

7. MRL + DMD 를 단일 manifest 로 결합
8. 누수 검증
9. 경로 유효성 표본 검사

## PART A — 진단: 그냥 합치면 무엇이 깨지는가

### 목적

- 두 데이터셋의 클래스 구성을 실제 숫자로 확인한다.
- "단순히 합치면 정확도가 오른다"가 성립하지 않는 구간인지 판정한다.

In [ ]:
# [셀 1] 설정 · 경로 (import·경로·시드는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}")
# --- 부트스트랩 끝 ---

# 프로젝트 모듈은 전부 src/ 에 평탄하게 둔다. 아직 패키지가 아니므로 sys.path 로 붙인다.
_src = str(config.PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import pandas as pd
import build_dmd_eye_dataset as B

SEED = 42

# 데이터셋 위치는 B.data_subdir 로 해석한다 (data/raw/<name> · data/<name> 둘 다 허용).
MRL_ROOT = B.data_subdir("MRL Eye") / "data"
MRL_MANIFEST = config.OUTPUTS_DIR / "mrl_split" / "mrl_manifest_subject.csv"
DMD_EYE_DIR = config.OUTPUTS_DIR / "dmd_eye"
EYE_DS_DIR = config.OUTPUTS_DIR / "eye_dataset"
EYE_DS_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_MANIFEST = EYE_DS_DIR / "eye_manifest.csv"

for _label, _p in [("MRL 이미지", MRL_ROOT), ("MRL manifest", MRL_MANIFEST),
                   ("DMD GT (STEP10)", config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv"),
                   ("YuNet", B.yunet_path())]:
    print(f"  [{'o' if Path(_p).exists() else 'X'}] {_label:16s} {config._rel(_p)}")

In [ ]:
# [셀 2] MRL·DMD 클래스 구성
mrl = pd.read_csv(MRL_MANIFEST)

mrl_tab = (mrl.groupby("split")
             .agg(subjects=("subject", "nunique"), images=("path", "size"),
                  closed=("class_idx", lambda s: int((s == 0).sum())))
             .reindex(["train", "val", "test"]))
mrl_tab["closed_pct"] = (mrl_tab.closed / mrl_tab.images * 100).round(1)
mrl_tab["glasses_pct"] = (mrl.groupby("split")["glasses"].mean() * 100).round(1)

dmd_gt = pd.read_csv(config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv")
dmd_plan = pd.DataFrame(B.plan())
dmd_gt_closed = int(dmd_plan.gt_closed.sum())
dmd_gt_open = int(dmd_plan.gt_open.sum())

print("=== MRL (subject 분할) ===")
print(mrl_tab)
print(f"\n=== DMD (프레임 단위 GT, 눈 2개 = crop 2장) ===")
print(f"close 프레임 {dmd_gt_closed:,} / open 프레임 {dmd_gt_open:,} "
      f"-> crop 상한 Closed {dmd_gt_closed*2:,} / Open {dmd_gt_open*2:,}")
print(f"DMD Closed 비율 : {dmd_gt_closed/(dmd_gt_closed+dmd_gt_open)*100:.1f}%"
      f"   (MRL 은 {mrl_tab.closed.sum()/mrl_tab.images.sum()*100:.1f}%)")

In [ ]:
# [셀 3] 단순 합본(stride 없이 전부)의 클래스 비율
mrl_c, mrl_o = int((mrl.class_idx == 0).sum()), int((mrl.class_idx == 1).sum())
dmd_c, dmd_o = dmd_gt_closed * 2, dmd_gt_open * 2

naive = pd.DataFrame([
    dict(source="MRL", Closed=mrl_c, Open=mrl_o),
    dict(source="DMD (전부)", Closed=dmd_c, Open=dmd_o),
    dict(source="단순 합본", Closed=mrl_c + dmd_c, Open=mrl_o + dmd_o),
])
naive["total"] = naive.Closed + naive.Open
naive["closed_pct"] = (naive.Closed / naive.total * 100).round(1)
naive["DMD_Open_share"] = (dmd_o / naive.total * 100).round(1)

print(naive.to_string(index=False))
print(f"\n단순 합본에서 DMD Open 이 전체의 {dmd_o/(mrl_c+mrl_o+dmd_c+dmd_o)*100:.1f}% 를 차지한다.")

> 이 표는 두 데이터셋을 조정 없이 합쳤을 때의 클래스 구성을 보여준다. MRL 단독에서는 Closed 가 절반 가까이지만 단순 합본에서는 크게 떨어지고, 그 감소분이 전부 DMD 의 Open 프레임에서 온다.

### 관찰 결과

- MRL 은 Closed 비율이 약 49%로 균형이며, 3개 split 의 안경 비율도 27~29%로 고르다.
- DMD 는 Closed 비율이 한 자릿수 후반대이고 Open 이 Closed 의 6배 이상이다.
- 단순 합본에서 Closed 비율이 MRL 단독보다 크게 낮아진다.
- 단순 합본에서 DMD 의 Open 프레임 한 종류가 전체 표본의 절반 이상을 차지한다.

## PART B — DMD 눈 crop 생성

### 목적

- DMD 를 학습에 넣되 MRL 의 클래스 균형을 깨지 않는 표본 수를 정한다.
- 학습에 쓰지 않은 subject 만 test 로 남겨 정직한 일반화 측정을 가능하게 한다.

### 결정 박스 2 — DMD 표본을 어떻게 줄일 것인가

- 문제: DMD 를 전부 넣으면 Open 이 전체를 지배해 Closed-Recall 이 떨어진다. (a) 클래스 가중치(`class_weight`)로 손실에서 보정할지, (b) Open 을 덜 뽑아 표본 단계에서 균형을 맞출지, (c) 프레임 stride 를 클래스와 무관하게 동일하게 줄지.
- 선택: **(b) 클래스별 비대칭 stride.** Closed 는 stride 2, Open 은 stride 12 로 뽑는다.
- 근거: DMD 의 Closed 는 희소해서(사용 가능 프레임의 한 자릿수 후반대) 촘촘히 모아야 절대량이 확보된다. 반면 Open 은 인접 프레임이 거의 동일한 중복 표본이라 12프레임(약 0.5초)마다 하나만 취해도 정보 손실이 작다. (a)는 튜닝 변수를 하나 더 늘리고 디코딩·저장 비용을 그대로 부담한다. (c)는 클래스 비율을 그대로 유지해 문제를 해결하지 못한다. stride 를 프레임 인덱스가 아니라 **클래스별 등장 순서**로 세는 이유는, 인덱스 기준으로 세면 짧은 Closed 구간이 통째로 빠지거나 남을 수 있기 때문이다.
- 사전 계획이다.

### 결정 박스 3 — DMD subject 를 어떻게 분할할 것인가

- 문제: DMD 13명을 train / val / test 로 나눠야 한다. 기존 코드는 test = {gC_13, gF_23, gZ_37, gB_6} 였다.
- 선택: **test = {gE_29, gC_13, gB_6, gZ_37}, val = {gA_5, gF_23}, train = 나머지 7명.**
- 근거: 기존 test 에는 안경 착용자가 0명이었다. DMD 안경 착용자는 gA_1 · gE_29 · gZ_36 3명뿐인데(STEP10 관찰), 전원이 train 에 들어가면 안경 조건에서 성능이 무너져도 test 가 잡아내지 못한다. 새 분할은 안경자를 train 2명 / test 1명으로 나누고, test 성별을 2:2 로 맞춘다. 같은 사람의 2세션(gB_10 · gZ_33 → train, gF_23 → val)은 반드시 같은 split 에 둔다. 대안(안경자를 test 에 2명 배치)은 train 의 안경 표본이 1명으로 줄어 학습이 불안정해진다.
- 사전 계획이다. 다만 안경자가 3명뿐이라는 제약 때문에 이 배분은 통계적 검정 대상이 아니라 **최소한의 방어 장치**로만 기능한다.

### 결정 박스 4 — test 에도 비대칭 stride 를 쓸 것인가

- 문제: train / val 과 같은 비대칭 stride 를 test 에도 적용하면 표본이 균형 잡혀 Accuracy 가 읽기 쉬워진다.
- 선택: **쓰지 않는다.** test 는 두 클래스에 같은 stride(4)를 적용한다.
- 근거: test 의 클래스 비율은 실전 비율(Closed 약 11%)을 반영해야 한다. 균형을 맞추면 Precision 이 실제보다 높게 나오고, PERCLOS(구간 내 Closed 시간 비율)의 기준값이 실제 운전 상황과 어긋난다. 균일 stride 는 표본 수만 줄이고 비율은 보존한다.
- 사전 계획이다.

In [ ]:
# [셀 4] 분할·stride 확정값 확인 (build_dmd_eye_dataset 의 상수를 그대로 읽는다)
split_tab = (pd.DataFrame([dict(subject=k, split=v) for k, v in B.DMD_SPLIT.items()])
             .merge(dmd_gt[["subject", "video", "glasses", "gender"]], on="subject"))

print("=== stride (split x 클래스) ===")
print(pd.DataFrame(B.STRIDE).T.rename_axis("split"))
print("\n=== subject 분할 ===")
print(split_tab.groupby("split")
      .agg(subjects=("subject", "nunique"), videos=("video", "size"),
           glasses_subjects=("glasses", lambda s: int(s.sum())))
      .reindex(["train", "val", "test"]))
print("\n=== 다중 세션 subject 가 한 split 에 있는지 ===")
multi = split_tab.groupby("subject").filter(lambda g: len(g) > 1)
print(multi.groupby("subject")["split"].nunique().to_dict(), "(모두 1이어야 한다)")
split_tab.sort_values(["split", "subject"])[["split", "subject", "gender", "glasses", "video"]]

In [ ]:
# [셀 5] 예상 표본 수 — 영상을 디코딩하기 전에 균형을 먼저 확인한다
plan_df = pd.DataFrame(B.plan())
agg = (plan_df.groupby("split")
       .agg(videos=("video", "size"), gt_closed=("gt_closed", "sum"),
            gt_open=("gt_open", "sum"), crops_closed=("crops_closed", "sum"),
            crops_open=("crops_open", "sum"))
       .reindex(["train", "val", "test"]))
agg["crops_total"] = agg.crops_closed + agg.crops_open
agg["closed_pct"] = (agg.crops_closed / agg.crops_total * 100).round(1)

print("=== DMD crop 예상 상한 (YuNet 검출 100% 성공 가정) ===")
print(agg)

# MRL train 과 합쳤을 때 균형이 유지되는지 확인
_mc = int(mrl.query("split=='train'").class_idx.eq(0).sum())
_mo = int(mrl.query("split=='train'").class_idx.eq(1).sum())
_dc, _do = int(agg.loc["train", "crops_closed"]), int(agg.loc["train", "crops_open"])
print(f"\n합본 train : Closed {_mc+_dc:,} / Open {_mo+_do:,}"
      f"  -> Closed {(_mc+_dc)/(_mc+_dc+_mo+_do)*100:.1f}%"
      f"  (MRL 단독 {_mc/(_mc+_mo)*100:.1f}%)")

> 이 표는 영상 디코딩 전에 계산한 split별 예상 crop 수와 클래스 비율이다. train·val 은 비대칭 stride 로 균형에 근접하고, test 는 균일 stride 로 실전 비율을 유지한다.

### 관찰 결과

- train·val 의 예상 Closed 비율이 절반 근처로 올라온다.
- test 의 예상 Closed 비율은 10% 대에 머문다.
- MRL train 과 합쳐도 합본 train 의 Closed 비율이 MRL 단독과 거의 같다.
- 다중 세션 subject 3명은 모두 단일 split 에 들어 있다.

### 목적

- 실제 crop 을 생성한다. 16개 영상 전체 디코딩이 필요해 이 셀만 시간이 오래 걸린다.
- 먼저 `BUILD_ALL = False` 로 한 영상만 돌려 동작을 확인하고, 정상이면 `True` 로 바꿔 전체를 만든다.

In [ ]:
# [셀 6] DMD 눈 crop 생성  ※ 실행 시간이 긴 셀
BUILD_ALL = False        # 동작 확인 후 True 로 바꿔 16개 전부 생성

jsons = B.dmd_jsons()
videos = jsons if BUILD_ALL else jsons[:1]
print(f"대상 영상 {len(videos)} / {len(jsons)}개"
      f"{'' if BUILD_ALL else '  (점검 모드 — 통합 manifest 를 만들려면 BUILD_ALL=True 필요)'}\n")

det = B.YuNetDetector()                      # 경로는 config.YUNET_MODEL. 02 와 같은 설정
dmd_manifest_path = B.build(videos, det, DMD_EYE_DIR)

### 관찰 결과

- 영상별 Closed / Open crop 수와 YuNet 얼굴 검출 실패 프레임 수가 출력된다.
- 검출 실패가 많은 영상은 이후 프레임 단위 평가(STEP13)에서 coverage 가 낮게 나올 후보다.

## PART C — 통합 manifest

### 목적

- MRL 과 DMD 를 **단일 manifest 1개**로 합쳐 split 정책을 하나로 만든다.
- 학습 소스를 바꾸는 실험(MRL only / DMD only / 합본)을 `source` 컬럼 필터만으로 처리할 수 있게 한다.

### 결정 박스 5 — 두 데이터셋을 어떤 형태로 결합할 것인가

- 문제: MRL 경로는 `data/MRL Eye/data` 기준 상대경로, DMD 경로는 `outputs/dmd_eye` 기준 상대경로다. (a) 매 실험마다 임시 CSV 를 만들어 절대경로로 합칠지, (b) 저장소 루트 기준 상대경로로 통일한 manifest 1개를 만들지.
- 선택: **(b) 저장소 루트 기준 상대경로 단일 manifest.** 로더에는 `mrl_root = str(config.PROJECT_ROOT)` 를 넘긴다.
- 근거: (a)는 실험마다 임시 파일이 생겨 재현이 어렵고, CSV 에 개인 PC 절대경로와 계정명이 남아 팀 공유에 부적합하다. (b)는 `mrl_dataset.make_dataset` 을 수정하지 않고 그대로 쓸 수 있다(로더는 `mrl_root` 를 앞에 붙일 뿐이다). MRL 의 기존 subject 분할은 그대로 승계하고 재분할하지 않는다 — 이미 누수 0 이 검증된 자산이다.
- 사전 계획이다.

In [ ]:
# [셀 7] MRL + DMD -> 통합 manifest
COLUMNS = ["path", "source", "subject", "video", "frame", "side",
           "label", "class_idx", "glasses", "split"]

_mrl_prefix = MRL_ROOT.resolve().relative_to(config.PROJECT_ROOT).as_posix()
mrl_u = pd.DataFrame({
    "path": _mrl_prefix + "/" + mrl["path"].astype(str),   # 저장소 루트 기준으로 통일
    "source": "mrl", "subject": mrl["subject"], "video": "-", "frame": -1, "side": "-",
    "label": mrl["label"], "class_idx": mrl["class_idx"],
    "glasses": mrl["glasses"].astype(int), "split": mrl["split"],
})

if dmd_manifest_path is None or not Path(dmd_manifest_path).exists():
    raise FileNotFoundError("DMD manifest 가 없습니다. 셀 6 을 BUILD_ALL=True 로 실행하세요.")
dmd_u = pd.read_csv(dmd_manifest_path)[COLUMNS]

if set(dmd_u.split.unique()) != {"train", "val", "test"}:
    raise ValueError(f"DMD split 이 3종이 아닙니다: {sorted(dmd_u.split.unique())}. "
                     "BUILD_ALL=True 로 16개 전부 생성해야 val/test 가 채워집니다.")

uni = pd.concat([mrl_u[COLUMNS], dmd_u], ignore_index=True)
uni.to_csv(UNIFIED_MANIFEST, index=False)

bal = (uni.groupby(["split", "source"])
       .agg(n=("path", "size"), closed=("class_idx", lambda s: int((s == 0).sum())),
            subjects=("subject", "nunique"))
       .reset_index())
bal["closed_pct"] = (bal.closed / bal.n * 100).round(1)
bal.to_csv(EYE_DS_DIR / "class_balance.csv", index=False)

print("저장 :", config._rel(UNIFIED_MANIFEST), f"({len(uni):,} rows)")
print()
print(bal.pivot(index="split", columns="source", values=["n", "closed_pct"])
      .reindex(["train", "val", "test"]))

> 이 표는 통합 manifest 의 split × source 별 표본 수와 Closed 비율이다. `source` 컬럼이 남아 있으므로 MRL only / DMD only / 합본 실험을 필터만으로 구성할 수 있고, split 정책은 하나뿐이다.

### 목적

- 학습·평가 사이에 정보가 새지 않는지 기계적으로 확인한다.
- 검증 결과를 CSV 로 남겨, 이후 노트북이나 발표 자료에서 같은 수치를 재인용할 수 있게 한다.

In [ ]:
# [셀 8] 누수 검증 — 하나라도 0 이 아니면 즉시 중단
checks = []

for src in ("mrl", "dmd"):
    sub = uni[uni.source == src]
    checks.append(dict(
        check=f"{src}: 2개 이상 split 에 등장하는 subject 수",
        value=int((sub.groupby("subject")["split"].nunique() > 1).sum())))

dmd_only = uni[uni.source == "dmd"]
checks.append(dict(check="dmd: 2개 이상 split 에 등장하는 video 수",
                   value=int((dmd_only.groupby("video")["split"].nunique() > 1).sum())))
checks.append(dict(check="dmd: 같은 (video, frame) 의 L/R 이 다른 split 에 있는 경우",
                   value=int((dmd_only.groupby(["video", "frame"])["split"]
                              .nunique() > 1).sum())))
checks.append(dict(check="mrl/dmd subject ID 충돌 수",
                   value=len(set(uni.query("source=='mrl'").subject)
                             & set(uni.query("source=='dmd'").subject))))
checks.append(dict(check="path 중복 수", value=int(uni.path.duplicated().sum())))
checks.append(dict(check="label 과 class_idx 불일치 수",
                   value=int(((uni.class_idx == 0) != (uni.label == "Closed")).sum())))

leak = pd.DataFrame(checks)
leak.to_csv(EYE_DS_DIR / "leakage_report.csv", index=False)
print(leak.to_string(index=False))

if leak.value.sum() != 0:
    raise AssertionError("누수 검증 실패. 위 표에서 0 이 아닌 항목을 확인하세요.")
print("\n누수 검증 통과 (모든 항목 0).")

In [ ]:
# [셀 9] 경로 유효성 표본 검사 — 전체 확인은 비용이 크므로 층화 표본으로 본다
sample = (uni.groupby(["split", "source"], group_keys=False)
          .apply(lambda g: g.sample(min(len(g), 150), random_state=SEED),
                 include_groups=True))
missing = [p for p in sample.path if not (config.PROJECT_ROOT / p).exists()]

print(f"표본 {len(sample)}개 중 존재하지 않는 경로 : {len(missing)}")
if missing:
    print("예시 :", missing[:3])
    raise FileNotFoundError("manifest 경로가 디스크와 어긋납니다.")

print("\n=== 통합 manifest 최종 ===")
print(uni.groupby("split").agg(n=("path", "size"), subjects=("subject", "nunique"),
                               closed_pct=("class_idx", lambda s: round((s == 0).mean()*100, 1)))
      .reindex(["train", "val", "test"]))

### 관찰 결과

- subject 가 2개 이상 split 에 등장하는 경우가 MRL·DMD 모두 0이다.
- DMD video 가 2개 이상 split 에 걸치는 경우가 0이고, 같은 프레임의 좌·우 눈이 서로 다른 split 으로 갈린 경우도 0이다.
- MRL subject ID(`s####`)와 DMD subject ID(`g?_#`)는 겹치지 않는다.
- 층화 표본에서 manifest 경로와 디스크 파일이 모두 일치한다.

## 해석

- split 정책이 하나로 줄어들면서, 기존에 세 곳(MRL 분할기 · DMD 생성기 · 학습 스크립트)에 나뉘어 있던 분할 결정이 통합 manifest 한 파일로 모였다. 학습 소스를 바꾸는 실험은 이제 `source` 컬럼 필터로만 표현되므로, 실험 조건이 달라져도 분할이 함께 바뀌는 일이 구조적으로 발생하지 않는다.
- 비대칭 stride 로 DMD 를 넣어도 합본 train 의 Closed 비율이 MRL 단독과 거의 같게 유지된다. 즉 이 합본은 "Open 을 늘려 Accuracy 를 올리는" 합본이 아니라 "Closed 표본에 RGB·실차 유사 조건을 추가하는" 합본이다. 이것이 Closed-Recall 개선으로 이어지는지는 STEP12 에서 측정한다.
- test 의 Closed 비율을 실전 비율로 남긴 것은 Precision 과 PERCLOS 기준값을 왜곡하지 않기 위한 선택이다. 그 대가로 test 에서 Accuracy 는 거의 의미가 없어지며, 주 지표는 Closed-Recall 이 된다.

## 한계

이 STEP 의 산출물은 데이터 구성의 균형과 누수 부재만 보증하며, 모델 성능 향상을 보증하지 않는다.

- **시간 상관**: stride 2 로도 같은 영상의 인접 프레임이 train 안에 다수 남는다. 누수는 아니지만 유효 표본 수가 실제보다 부풀려진다. Closed 연속 구간이 짧아 stride 를 더 키우면 Closed 표본이 붕괴하므로 감수한 선택이다.
- **좌우 눈 상관**: 같은 프레임의 L·R crop 은 독립 표본이 아니다. 학습에서는 증강 효과가 있으나, 평가 지표는 프레임 단위로 집계해야 한다(STEP12에서 처리).
- **안경 표본**: DMD 안경 착용자 3명(train 2 / test 1). 안경 조건의 성능 차이를 통계적으로 주장할 수 없다.
- **도메인 차이**: MRL 은 IR 그레이스케일, DMD 는 RGB 다. `grayscale=True` 로 채널을 통일해도 조명·각도·센서 차이는 남는다.
- **검출기 의존**: DMD crop 은 YuNet 검출 성공 프레임에서만 나온다. 검출이 어려운 자세·조명이 표본에서 체계적으로 빠질 수 있다.
- **연출 상황**: DMD 는 전부 `Car Stopped` 다(STEP10). 실제 주행 졸음이 아니다.

## STEP11 요약

### Takeaway

- MRL 과 DMD 를 조정 없이 합치면 DMD 의 Open 프레임 하나가 전체 표본의 절반 이상을 차지해 Closed 비율이 무너진다. "합치면 좋아진다"가 성립하지 않는 구간임을 수치로 확인했다.
- 클래스별 비대칭 stride(Closed 2 / Open 12)로 DMD 를 뽑으면 합본 train 의 Closed 비율이 MRL 단독과 거의 같게 유지된다.
- DMD subject 분할을 재구성해 test 에 안경 착용자를 1명 포함시켰다. 기존 분할은 test 안경자가 0명이어서 안경 일반화를 측정할 수 없었다.
- test 는 균일 stride 로 실전 Closed 비율을 보존했다. 주 지표는 Accuracy 가 아니라 Closed-Recall 이다.
- split 정책이 3개에서 1개(통합 manifest)로 줄었고, 누수 검증 6개 항목이 모두 0이다.